# Обучение модели распознавания растений


## Data

In [ ]:
!pip install torch torchvision opencv-python matplotlib scikit-learn kagglehub onnx onnxruntime tqdm seaborn

In [2]:
import kagglehub
import shutil
import os

tmp_path = kagglehub.dataset_download("abdallahalidev/plantvillage-dataset")
source_dir = os.path.join(tmp_path, 'plantvillage dataset', 'color')
print("Path to dataset files:", tmp_path)

target_dir = 'data/color'
os.makedirs(target_dir, exist_ok=True)

if os.path.exists(source_dir):
    if os.path.exists(target_dir):
        shutil.rmtree(target_dir)

    shutil.move(source_dir, target_dir)
    print(f"Готово! Данные теперь в {target_dir}")
else:
    print("Ошибка: Папка 'color' не найдена по пути:", source_dir)
    print("Содержимое архива:", os.listdir(source_dir))


Using Colab cache for faster access to the 'plantvillage-dataset' dataset.
Path to dataset files: /kaggle/input/plantvillage-dataset


KeyboardInterrupt: 

Доступно 38 папок с названием в виде <Название растения>__<healthy/название болезни>. В каждой папке по одному изображению размером 256 на 256 пикселей.

## EDA

In [ ]:
from torchvision import datasets, transforms

transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
])

full_dataset = datasets.ImageFolder(root='data/color', transform=transform)

In [ ]:
print(full_dataset.classes)

## Обучение

## Предсказание

## Графики и анализ

## 1. Импорт библиотек

In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as T
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# Если планируете использовать Grad-CAM, раскомментируйте установку:
# !pip install grad-cam

## 2. Визуализация Bounding Box и расчет IoU

IoU (Intersection over Union) используется для оценки качества локализации (где именно находится больной лист на фото). Это отличная визуализация для презентаций!

In [ ]:
def calculate_iou(boxA, boxB):
    # box: [x1, y1, x2, y2]
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])

    interArea = max(0, xB - xA) * max(0, yB - yA)

    boxAArea = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    boxBArea = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])

    iou = interArea / float(boxAArea + boxBArea - interArea)
    return iou

def visualize_iou(image_path, gt_box, pred_box):
    """
    gt_box - Ground Truth (истинный бокс от разметчика, ЗЕЛЕНЫЙ)
    pred_box - Предсказание нейронки (КРАСНЫЙ)
    """
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    iou = calculate_iou(gt_box, pred_box)

    # Рисуем предсказание (Красный)
    cv2.rectangle(img, (int(pred_box[0]), int(pred_box[1])),
                  (int(pred_box[2]), int(pred_box[3])), (255, 0, 0), 3)
    cv2.putText(img, f'Pred', (int(pred_box[0]), int(pred_box[1])-10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 0, 0), 2)

    # Рисуем истину (Зеленый)
    cv2.rectangle(img, (int(gt_box[0]), int(gt_box[1])),
                  (int(gt_box[2]), int(gt_box[3])), (0, 255, 0), 3)
    cv2.putText(img, f'GT (IoU: {iou:.2f})', (int(gt_box[0]), int(gt_box[3])+30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)

    plt.figure(figsize=(8, 8))
    plt.imshow(img)
    plt.axis('off')
    plt.title("IoU Visualization")
    plt.show()

## 3. Explainable AI: Тепловые карты (Grad-CAM)
Grad-CAM показывает, на какие части изображения модель обратила внимание при предсказании класса. Это особенно важно для диагностики болезней (убедиться, что модель смотрит на пятна, а не на задний фон).

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

def show_gradcam(model, target_layer, img_tensor, original_img):
    cam = GradCAM(model=model, target_layers=[target_layer])

    # Предсказываем для самого вероятного класса (или укажите ClassifierOutputTarget(class_idx))
    targets = [ClassifierOutputTarget(2)] # Пример: класс 2

    grayscale_cam = cam(input_tensor=img_tensor, targets=targets)[0, :]

    # Наложение
    img_normalized = np.float32(original_img) / 255
    visualization = show_cam_on_image(img_normalized, grayscale_cam, use_rgb=True)

    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    plt.imshow(original_img)
    plt.title('Original Image')
    plt.axis('off')

    plt.subplot(1, 2, 2)
    plt.imshow(visualization)
    plt.title('Grad-CAM Heatmap')
    plt.axis('off')
    plt.show()

## 4. Заготовка для K-Fold Validation
Если у вас небольшой датасет и вы хотите использовать K-Fold (например, $K=5$).

In [ ]:
from sklearn.model_selection import KFold
from torch.utils.data import Subset, DataLoader

# Предположим, у нас есть full_dataset (экземпляр Dataset)
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_ids, val_ids) in enumerate(kfold.split(np.arange(100))): # 100 - длина датасета
    print(f'FOLD {fold}')
    print('--------------------------------')

    # train_sub = Subset(full_dataset, train_ids)
    # val_sub = Subset(full_dataset, val_ids)

    # trainloader = DataLoader(train_sub, batch_size=32, shuffle=True)
    # valloader = DataLoader(val_sub, batch_size=32, shuffle=False)

    # Дальше идет инициализация модели и стандартный цикл обучения (epochs)
    # ...

    break # Для примера пройдем только 1 фолд

## 5. Экспорт модели в ONNX для использования в Production
ONNX не требует PyTorch в production-среде (например, в Docker-контейнере с FastAPI).

In [ ]:
def export_to_onnx(model, save_path="weights/plant_model.onnx"):
    model.eval()

    # Создаем фиктивный тензор той же размерности, что и вход модели
    # Например: Batch_Size=1, Channels=3, H=224, W=224
    dummy_input = torch.randn(1, 3, 224, 224, device='cpu')

    torch.onnx.export(
        model,
        dummy_input,
        save_path,
        export_params=True,
        opset_version=14,          # Opset версия (11-14 обычно безопасны)
        do_constant_folding=True,  # Оптимизация графа
        input_names=['input'],     # Имя входного узла
        output_names=['output'],   # Имя выходного узла
        dynamic_axes={
            'input': {0: 'batch_size'},
            'output': {0: 'batch_size'}
        } # Поддержка динамического размера батча
    )
    print(f"Модель успешно экспортирована в {save_path}!")

# Пример вызова:
# model = torchvision.models.resnet18(pretrained=True)
# export_to_onnx(model)